In [2]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
from pathlib import Path

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

*** Device() ***
device = cuda:7



In [3]:
def sort(df, metric:str, ascending:bool=False):
    # Set MultiIndex
    multi_df = df.set_index(['config', 'metric']).sort_index()

    # Extract sorting values based on the given metric
    sort_values = multi_df.xs(metric, level='metric')['mean']

    # Sort configs based on the sort_metric's mean
    sorted_configs = sort_values.sort_values(ascending=ascending).index

    # Reindex the dataframe to reflect sorted order
    sorted_multi_df = multi_df.reindex(sorted_configs, level='config')

    return sorted_multi_df

Classifiers
---

In [4]:
import numpy as np

In [5]:
##### get data ####

data = d.Data(
    # datasets
    tcga_project = 'TCGA-HNSC',
    metadata_subtype_col = 'paper_RNA',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':[np.nan, 'Primary Tumor', 'Metastatic']},
)


**** Data() ****
log0_method      log1p            str
class_weights    (5,)             Tensor (cuda:7)
gene_counts      (4383, 323)      DataFrame
metadata         (323, 3)         DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (323, 4383, 1)   Tensor (cuda:7)
y                (323, 5)         Tensor (cuda:7)
y_labels         5                list
num_samples      323              int
num_nodes        4383             int
num_features     1                int
num_labels       5                int
num_masks        305              int



In [6]:
data.y_labels

['Solid Tissue Normal', 'Basal', 'Mesenchymal', 'Atypical', 'Classical']

In [7]:
#### get configs ####

# classifiers
MLPc = t.Config(name='MLPc')
MLPc.add(
    key='classifier',
    model_class=m.MLPClassifier,
    model_kwargs={
        'in_features':data.num_nodes,
        'out_features':data.num_labels,
        'flatten':True,
    },
    trainer_class=t.ClassificationTrainer,
    trainer_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'verbose':False,
    }
)

GCNc = t.Config(name='GCNc')
GCNc.add(
    key='classifier',
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':data.num_features,
        'out_features':data.num_labels,
        'relation':data.relation,
        'num_nodes':data.num_nodes
    },
    trainer_class=t.ClassificationTrainer,
    trainer_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'verbose':False,
    }
)

GATc = t.Config(name='GATc')
GATc.add(
    key='classifier',
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':data.num_features,
        'out_features':data.num_labels,
        'relation':data.relation,
        'num_nodes':data.num_nodes
    },
    trainer_class=t.ClassificationTrainer,
    trainer_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'verbose':False,
    }
)

c_configs = [MLPc, GCNc, GATc]

In [8]:
num_trials = 100
num_epochs = 25

# define experiment, run experiment
classifiers = t.Experiment(
    num_trials=num_trials,
    num_epochs=num_epochs,
    X=data.X,
    y=data.y,
    generator=generator,
)

# add configs
for _config in c_configs:
    classifiers.add_config(_config)

# run exp
classifiers.run_experiment(comment=f'GBM_classifiers_{num_trials}t_{num_epochs}e', verbose=True)

100%|██████████| 100/100 [06:59<00:00,  4.20s/it]


In [9]:
sort(classifiers.summary['classifier'], 'accuracy')

mean       std        ci
config metric                                 
GATc   accuracy   0.886250  0.045434  0.009015
       f1         0.886062  0.045850  0.009098
       loss       1.615197  0.621950  0.123408
       precision  0.898625  0.042479  0.008429
       recall     0.886250  0.045434  0.009015
GCNc   accuracy   0.884583  0.045530  0.009034
       f1         0.884327  0.046011  0.009130
       loss       1.626465  0.666052  0.132159
       precision  0.896642  0.041535  0.008242
       recall     0.884583  0.045530  0.009034
MLPc   accuracy   0.881667  0.042488  0.008430
       f1         0.881341  0.042202  0.008374
       loss       1.647393  0.636469  0.126289
       precision  0.894781  0.037682  0.007477
       recall     0.881667  0.042488  0.008430

Autoencoders
---

In [10]:
#### get configs ####

MLPae = t.Config(name='MLPae')
MLPae.add(
    key='autoencoder',
    model_class=m.MLPClassifier,
    model_kwargs={
        'in_features':data.num_nodes,
        'out_features':data.num_labels,
        'flatten':True,
    },
    trainer_class=t.RegressionTrainer,
    trainer_kwargs={
        'loss_fn':nn.L1Loss(),
        'verbose':False,
    }
)

GCNae = t.Config(name='GCNae')
GCNae.add(
    key='autoencoder',
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':data.num_features,
        'out_features':data.num_labels,
        'relation':data.relation,
        'num_nodes':data.num_nodes
    },
    trainer_class=t.RegressionTrainer,
    trainer_kwargs={
        'loss_fn':nn.L1Loss(),
        'verbose':False,
    }
)

GATae = t.Config(name='GATae')
GATae.add(
    key='autoencoder',
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':data.num_features,
        'out_features':data.num_labels,
        'relation':data.relation,
        'num_nodes':data.num_nodes
    },
    trainer_class=t.RegressionTrainer,
    trainer_kwargs={
        'loss_fn':nn.L1Loss(),
        'verbose':False,
    }
)

ae_configs = [MLPae, GCNae, GATae]

In [11]:
num_trials = 100
num_epochs = 60

# define experiment, run experiment
autoencoders = t.Experiment(
    num_trials=num_trials,
    num_epochs=num_epochs,
    X=data.X,
    y=data.y,
    generator=generator,
)

# add configs
for _config in ae_configs:
    autoencoders.add_config(_config)

# run exp
autoencoders.run_experiment(comment=f'GBM_autoencoders_{num_trials}t_{num_epochs}e', verbose=True)

100%|██████████| 100/100 [26:24<00:00, 15.84s/it]


In [12]:
sort(autoencoders.summary['autoencoder'], 'mae', ascending=True)

mean       std        ci
config metric                              
GATae  corr    0.641688  0.072655  0.014416
       mae     0.254159  0.049803  0.009882
       mse     0.115022  0.050270  0.009975
       rmse    0.333855  0.059991  0.011904
GCNae  corr    0.642831  0.074208  0.014724
       mae     0.254276  0.047255  0.009376
       mse     0.114351  0.042716  0.008476
       rmse    0.333613  0.055536  0.011020
MLPae  corr    0.632446  0.085914  0.017047
       mae     0.260243  0.059844  0.011874
       mse     0.121329  0.059519  0.011810
       rmse    0.341337  0.069759  0.013842